In [44]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
from sklearn.preprocessing import StandardScaler

In [45]:
Data = pd.read_csv('../data/processed_data.csv')
df = pd.DataFrame(Data)

In [46]:
def Logreg(X, y, Testsize, solvers=['liblinear', 'lbfgs', 'newton-cg', 'sag', 'saga']):
    eval_list = []
    y_pred = None

    for x in Testsize:
        # ۱. تقسیم داده‌ها
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=x, random_state=0
        )

        # ۲. مقیاس‌بندی داده‌ها (اضافه شده برای رفع ارور)
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        for solver_name in solvers:
            try:
                logreg = LogisticRegression(
                    solver=solver_name,
                    class_weight='balanced',
                    max_iter=1000,
                    random_state=0
                )
                # ۳. آموزش مدل روی داده‌های اسکیل‌شده
                logreg.fit(X_train_scaled, y_train)

                # ۴. پیش‌بینی روی داده‌های تست اسکیل‌شده
                y_pred = logreg.predict(X_test_scaled)

                acc = metrics.accuracy_score(y_test, y_pred)
                score = logreg.score(X_test_scaled, y_test)

                eval_list.append({
                    'Test_size': x,
                    'Solver': solver_name,
                    'acc': acc,
                    'score': score
                })
            except Exception as e:
                print(f"Solver {solver_name} with test_size {x} failed: {e}")

    df_evaluation = pd.DataFrame(eval_list)

    return X_train, X_test, y_train, y_test, y_pred, df_evaluation

In [47]:
def highlight_max(s):
    is_max = s == s.max()
    return ['background-color: lightgreen' if v else '' for v in is_max]

In [48]:
X = df.drop(columns=['Personal Loan', 'Place', 'ID'], errors='ignore')
X = pd.get_dummies(X, drop_first=True)
y = df['Personal Loan']

In [49]:
X_train, X_test, y_train, y_test, y_pred, df_eval = Logreg(X, y, [0.2, 0.25, 0.3])

df_eval.style.apply(highlight_max, subset=['acc', 'score'])

,Test_size,Solver,acc,score
0,0.200000,liblinear,1.000000,1.000000
1,0.200000,lbfgs,1.000000,1.000000
2,0.200000,newton-cg,1.000000,1.000000
3,0.200000,sag,1.000000,1.000000
4,0.200000,saga,1.000000,1.000000
5,0.250000,liblinear,1.000000,1.000000
6,0.250000,lbfgs,1.000000,1.000000
7,0.250000,newton-cg,1.000000,1.000000
8,0.250000,sag,1.000000,1.000000
9,0.250000,saga,1.000000,1.000000
